# Section 4: Vibe Coding in Practice + Product Design

> **Estimated time: 90–110 minutes**
> **Goal:** Use pure natural language to have AI generate a PDF summary tool (Vibe Coding), then complete the SmartLearn Agent product design document.
> **Done when:** `pdf_summary.py` runs successfully, and `docs/design.md` contains user stories, a feature list, and a data flow design.
> **Prerequisites:** Completed [Section 3](Section3_CLI_QA_Tool.ipynb)

---

> **Note: Copy the commands below into a terminal and run them there. Do not run code directly in this Notebook.**

## 4.1 Vibe Coding Is a Controlled Loop (5 min)

Vibe Coding means expressing software intent in natural language and letting an agent perform much of the implementation. It is fast because Claude Code can read files, edit code, and run commands — which also means a vague instruction can create a large, incorrect diff very quickly.

### The six-stage loop we will practice

```text
PRD → Plan → Implement one slice → Test → Debug from evidence → Iterate
       ↑                                                        │
       └───────────── review diff / narrow scope ────────────────┘
```

| Stage | Student responsibility | Evidence before moving on |
|---|---|---|
| PRD | Define goal, input/output, constraints, done when | Another student can decide pass/fail |
| Plan | Catch assumptions and scope creep | Expected files and tests are explicit |
| Implement | Approve one bounded slice | Diff contains only intended files |
| Test | Try happy path and edge cases | Commands and observed outputs are recorded |
| Debug | Provide full evidence | Root cause + smallest fix + rerun command |
| Iterate | Add one improvement | New test passes and old test still passes |

### Good fit vs poor fit

Use this workflow for prototypes, learning projects, and small tools where you can review and test the output. Do not blindly use it for credentials, payments, destructive data operations, or code you cannot verify.

> Today's goal is not 'AI created a file'. The goal is to finish one controlled loop and collect evidence at every checkpoint.

---

## 4.2 Verify the Workshop AI Coding Tool (2 min)

This workshop uses one fixed coding workflow: **Claude Code calls DeepSeek directly**. It can plan steps, create files, revise code, and run approved commands from the terminal.

The two services have different jobs:

| Service | Used for | Connection |
|------|----------|------------|
| Claude Code + DeepSeek | AI coding, planning, editing, and debugging | `https://api.deepseek.com/anthropic` |
| OpenRouter | Learning API calls and running model/prompt comparisons in Python | `https://openrouter.ai/api/v1` |

From the project root, run `claude`, then `/model`; confirm DeepSeek before continuing.

> **Boundary:** OpenRouter does not sit between Claude Code and DeepSeek. It remains a separate teaching API.

---

## 4.3 Write a Mini PRD (5 min)

### What Is a PRD?

PRD stands for **Product Requirements Document**. In professional software development, a product manager writes a PRD to describe what to build, why to build it, and what the result should look like.

In Vibe Coding, the PRD is the task briefing you hand to the AI. The clearer it is, the closer the generated code will match expectations.

### Create the PRD File

In the project folder, create a file named `pdf_summary_prd.md` with the following content:

In [ ]:
%%writefile pdf_summary_prd.md
# PDF Summary Tool — Mini PRD

## Goal
A CLI tool that reads a PDF file and prints a structured summary.

## Usage
```
python3 pdf_summary.py <path-to-pdf>
```

## Requirements
1. Accept a PDF file path as a command-line argument
2. Extract text from the PDF
3. Send extracted text to an LLM through OpenRouter
4. Print exactly three sections: Overview, Key Points, and Limitations
5. Every key point must include a [Page X] citation
6. Never print the API key or PDF contents during normal operation

## Tech Constraints
- Use `python-dotenv` to load API keys from `.env`
- Use `openai` SDK with OpenRouter as the base URL
- PDF library: (let AI decide the best option)

## Done When
1. `python3 -m py_compile pdf_summary.py` succeeds
2. A short text-based PDF produces all three output sections and page citations
3. A missing path prints a friendly usage/error message without a traceback
4. A scanned PDF with no extractable text explains the limitation instead of calling the LLM with empty text
5. `git status --short` does not show `.env`

> **Expected output:**
> ```
> Writing pdf_summary_prd.md
> ```

Once the file is created, `pdf_summary_prd.md` appears in the project directory.

### PRD Writing Tips

Note several features of this PRD:

1. **Clear Goal:** one sentence stating what to build
2. **Concrete Usage:** gives the exact command format, and AI will write code to match it
3. **Tech Constraints pin down the stack:** `python-dotenv` and the `openai` SDK are specified, so AI will follow that choice
4. **Deliberate decision:** the PDF library is left unspecified, but Claude Code must explain the choice
5. **Edge cases:** missing files and scanned PDFs are defined before implementation
6. **Mechanical checks:** syntax and Git safety can be verified with exact commands

✅ **Checkpoint 1:** `pdf_summary_prd.md` exists and contains all five parts: Goal, Usage, Requirements, Tech Constraints, Done When.

---

## 4.4 Run the PRD → Plan → Implement Loop (15 min)

This is the heart of controlled Vibe Coding: Claude Code does the implementation work, while you control scope and acceptance.

### Steps

#### Step 1: Start Claude Code

In the project root, run `git status --short`, then `claude`. Use `/model` to confirm DeepSeek. A clean or understood Git status is your rollback checkpoint.

#### Step 2: Ask for a plan first

Enter this prompt:

```text
Read CLAUDE.md and pdf_summary_prd.md. Plan only; do not edit files or run install commands.
Return exactly:
1. Requirements and acceptance tests you understood
2. Files you expect to create or modify
3. PDF library choice with one reason and one limitation
4. Small implementation steps
5. Risks or ambiguities
```

![placeholder: Claude Code terminal showing the implementation plan]

#### Step 3: Review the plan like a teacher

Reject or revise the plan if it adds a web UI, database, vector store, login, Docker, OCR, or unrelated refactor. None of those are in the PRD.

> **Teacher check:** ask one student to name every planned file and another student to map each file to a PRD requirement.

#### Step 4: Authorize one implementation slice

Send:

```text
Implement the approved plan. Keep changes limited to pdf_summary.py and requirements documentation if needed.
Do not read, print, or modify .env. Do not add a UI, database, vector store, OCR, or deployment files.
After editing, run python3 -m py_compile pdf_summary.py and show the changed files and result.
```

#### Step 5: Watch the AI work

After sending, observe what the AI does:

1. **Planning:** AI analyzes the requirements and lists the implementation steps
2. **Creating files:** AI creates `pdf_summary.py` and writes the code step by step
3. **Choosing dependencies:** AI picks a PDF processing library (usually `pdfplumber` or `PyMuPDF`)
4. **Possible extras:** AI may also create a `.env.example` file describing which environment variables to configure

![placeholder: Claude Code terminal creating pdf_summary.py and showing a diff]

#### Step 6: Review dependency commands before approving

After generating the code, AI usually tells you which Python packages to install. If AI chose `pdfplumber` (the most common choice), run in the terminal:

In [ ]:
!pip install pdfplumber

> **Expected output:**
> ```
> Successfully installed pdfplumber-x.x.x pdfminer.six-x.x.x ...
> ```

If AI chose `PyMuPDF`, run instead:

In [ ]:
!pip install PyMuPDF

> **Expected output:**
> ```
> Successfully installed PyMuPDF-x.x.x
> ```

Also make sure `openai` and `python-dotenv` from earlier sections are installed:
If AI chose `pypdf`, run `pip install pypdf` instead. Install only the PDF library used by `pdf_summary.py`.


In [ ]:
!pip install openai python-dotenv

> **Expected output:**
> ```
> Requirement already satisfied: openai in ...
> Requirement already satisfied: python-dotenv in ...
> ```
>
> (If these were installed in earlier exercises, you will see `Requirement already satisfied`, which is normal.)

✅ **Checkpoint 2:** `pdf_summary.py` exists, `python3 -m py_compile pdf_summary.py` passes, the diff is limited to expected files, and every installed dependency is understood.

> **Troubleshooting:**
> - `claude` is not found → reinstall it with `npm install -g @anthropic-ai/claude-code` and reopen the terminal
> - Import errors in the generated code → A dependency is probably missing; run `pip install` as the AI suggests
> - AI chose a different PDF library → That is fine; install it with the command AI provides. The libraries behave similarly

---

## 4.5 Run a Small Acceptance Test Matrix (10 min)

Do not test only the happy path. Run the cheapest failures first, then spend an API call on a valid PDF. Record pass/fail beside each row.

### Test 1: Missing path (no API call expected)

Run `python3 pdf_summary.py does-not-exist.pdf`. Pass if the program prints a friendly error without a Python traceback.

### Test 2: Short text-based PDF

Find a PDF file on the computer to test with. For example:
- Lecture slides from a course
- A paper downloaded earlier
- Any document saved as PDF

> Use a short PDF under 10 pages with selectable text. A lecture handout is better than a random 100-page paper because you can verify the summary yourself.

Pass if the output contains `Overview`, `Key Points`, `Limitations`, and at least one `[Page X]` citation.

### Test 3: Scanned or image-only PDF (optional)

Pass if the tool explains that no extractable text was found and does not send an empty prompt to the LLM.

### Run the valid-PDF command

Run in the terminal (replace the path with the actual PDF file path):

In [ ]:
!python3 pdf_summary.py /path/to/your/file.pdf

> **Expected output:**
>
> The wording varies, but the structure must match the PRD:
>
> ```
> ## Overview
> A short summary of the document.
>
> ## Key Points
> - A verifiable point from the document [Page 1].
> - Another point from the document [Page 2].
>
> ## Limitations
> Missing context, extraction limits, or other caveats.
> ```
>
> Pass only if all three headings appear and every Key Points bullet ends with a `[Page X]` citation.


---

## 4.6 Review the AI-Generated Code (10 min)

The tool works, but Vibe Coding does not end here. You must review the code AI wrote — **AI writes the first draft; you are the editor.**

Open `pdf_summary.py` and answer the following four questions:

### Question 1: Which PDF library did AI choose?

Find the `import` statements at the top of the code. You will see something like:
```python
import pdfplumber
```
or:
```python
import fitz  # PyMuPDF
```

Recall that the PRD deliberately left the PDF library unspecified. Claude Code made a proposal; your job is to verify the choice and understand its limitation before accepting it.

### Question 2: How does it handle multi-page PDFs?

Find the text extraction part. There is usually a loop over all pages:
```python
for page in pdf.pages:
    text += page.extract_text()
```

Think: does it concatenate the text of all pages and send it to the LLM in one shot?

### Question 3: Is the API key loaded from `.env`?

Find the code that loads the API key. You should see:
```python
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")
```

This is required by the Tech Constraints in the PRD. If AI ignored the requirement (for example, hard-coded the API key), ask AI to fix it.

### Question 4: What happens with a 500-page PDF?

This is the most important review question. A 500-page PDF may contain hundreds of thousands of words. Most LLM APIs have a **token limit** (for example, 128K tokens); text beyond the limit gets truncated or triggers an error.

Check whether the code has:
- Text length limits or truncation logic
- Chunked processing
- Error handling (a helpful message when the text is too long)

Most likely none of these exist — a classic Vibe Coding blind spot: **AI usually handles only the happy path; humans must guard the edge cases.**

### Mechanical review commands

Run `git diff --stat`, `git diff`, and `git status --short`. Confirm that `.env` is not shown, no API key appears in the diff, and no unrelated file was created.

✅ **Checkpoint 4:** You can explain the code path, library choice, API key handling, large-PDF limitation, and every changed file in the Git diff.

> **Key insight:**
> AI-written code usually runs, but "runs" and "reliable" are two different things. Code review closes the gap between AI output and production-grade code.

---

## 4.7 Iterate in Natural Language (5 min)

Another strength of Vibe Coding: **you can modify existing code with natural language requests.** Tell AI what to change, and it locates and edits the relevant code automatically.

Pick **one** improvement. Before editing, ask Claude Code to name the files and tests it will touch. One change at a time keeps the diff explainable.

### Option A: Add a page range flag

```
Add a --pages START-END flag to pdf_summary.py only.
Example: python3 pdf_summary.py my.pdf --pages 1-5
Reject malformed ranges with a friendly message. Do not change the default behavior.
After editing, test one valid range, one invalid range, and the original no-flag command.
```

### Option B: Switch the output to bullet points

```
Change only the LLM output contract so Key Points uses 3-5 bullets.
Every bullet must end with [Page X]. Keep Overview and Limitations unchanged.
After editing, rerun the same PDF and check all three sections.
```

### Option C: Show extraction progress

```
Add extraction progress in the format: Extracting page 3/10...
Do not print extracted PDF text or the API key.
After editing, test a multi-page PDF and the missing-file case.
```

### Watch How AI Modifies the Code

After sending, observe:

1. AI locates the parts of `pdf_summary.py` that need changes
2. It modifies only the relevant code and leaves the rest untouched
3. It runs the new acceptance case and one regression case
4. You inspect the diff before accepting the checkpoint

![placeholder: Claude Code showing the focused diff for an improvement]

When the change is done, run the test again:

In [ ]:
!python3 pdf_summary.py /path/to/your/file.pdf

> **Expected output:**
>
> The wording varies, but the structure must match the PRD:
>
> ```
> ## Overview
> A short summary of the document.
>
> ## Key Points
> - A verifiable point from the document [Page 1].
> - Another point from the document [Page 2].
>
> ## Limitations
> Missing context, extraction limits, or other caveats.
> ```
>
> Pass only if all three headings appear and every Key Points bullet ends with a `[Page X]` citation.


---

## 4.8 Practical Comparison: Uncontrolled vs Controlled Vibe Coding (5 min)

| Moment | Uncontrolled prompt | Controlled workflow |
|---|---|---|
| Start | `Build a PDF tool` | PRD with input, output, constraints, and acceptance tests |
| Before edits | Agent starts immediately | Plan-only response lists files, tests, and risks |
| Scope | UI/database/vector store may appear | Explicit exclusions prevent scope creep |
| Success | 'It generated code' | Syntax + happy path + edge cases pass |
| Failure | `It doesn't work, fix it` | Command + traceback + expected/actual + recent change |
| Iteration | Multiple features in one prompt | One feature, one focused diff, regression test |
| Safety | Hope secrets are untouched | `CLAUDE.md`, `.gitignore`, diff review, clean checkpoint |

### Student exit ticket

Answer in your own words:

1. Which acceptance test caught a problem that the happy-path demo would miss?
2. Which line in your prompt prevented scope creep?
3. What changed between the first and final Git diff?
4. What part of `pdf_summary.py` do you still not understand? Ask Claude Code to explain that part without editing files.

A Vibe Coding task is complete only when you can explain the behavior, show passing evidence, and recover from a bad change.

---

## 4.9 Commit the Code

Before committing, rerun the original valid-PDF test and the edge case affected by your iteration. Then inspect `git diff --stat` and `git diff`. Commit only when the evidence matches the PRD.

Commit the files from this section to Git:

In [ ]:
!git add pdf_summary.py pdf_summary_prd.md
!git commit -m "feat: PDF summary tool (vibe coded)"
!git push

> **Expected output:**
> ```
> [main abc1234] feat: PDF summary tool (vibe coded)
>  2 files changed, XX insertions(+)
>  create mode 100644 pdf_summary.py
>  create mode 100644 pdf_summary_prd.md
> ```

> **Troubleshooting:**
> - `nothing to commit` → Check that the files are inside the project folder. Run `git status` to see the current state
> - `push` fails → Confirm the remote repository exists on GitHub and was added with `git remote add origin <url>`

✅ **Checkpoint 6:** `pdf_summary.py` and `pdf_summary_prd.md` are committed and pushed to GitHub.

---

# Part 2: Product Design

> The coding tool is done. One last step: plan tomorrow's project.

## 4.10 Product Design Overview

### Background

On Days 2–3 we will build the complete **SmartLearn Agent** — an AI study assistant with a web interface that parses PDF lecture slides and answers student questions.

### Why Design First

The most common beginner mistake is jumping straight into code. It is like building a house without a blueprint — halfway through the walls you realize the windows are in the wrong place and have to tear them down.

**Product design answers three questions:**

1. **Who is it for?** → Who are the target users, and what do they need most
2. **What does it do?** → Which features to build, and in what order
3. **How does it work?** → How data flows through the system, and how the modules cooperate

Answer these three questions on paper (or in a document) first, and coding gains a clear direction with far less rework.

### Deliverable for This Section

A `docs/design.md` file containing:
- **User Stories:** what users need
- **Feature List:** features ranked by priority
- **Data Flow Diagram:** how data moves through the system

---

## 4.11 User Stories

### What Is a User Story?

A **user story** is a standard format for describing requirements in product design. Each user story states in one sentence who wants to do what, and why.

The fixed format:

```
As a [role], I want to [do something], so that [benefit].
```

**Why this format?** It forces you to think from the user's perspective. "I want to add a button" is technical thinking; "a student wants one-click PDF upload" is user thinking.

### An Example

For SmartLearn Agent:

> As a **student**, I want to **upload a PDF and ask questions about it**, so that **I can study more efficiently**.

This user story tells us:
- The user is a **student**
- They want to **upload a PDF and ask questions**
- The goal is **more efficient studying**

### Hands-on: Create `docs/design.md`

#### Step 1: Create the `docs` directory

In the terminal, make sure you are at the root of the `smartlearn-agent` project, then run:

In [ ]:
mkdir -p docs

> **Expected output:** no output (silent success). The `-p` flag means "skip without error if the directory already exists".

#### Step 2: Create the `docs/design.md` file

Create `docs/design.md` in your editor, or ask Claude Code to create an empty file without changing anything else.

Copy the template below into it:

```markdown
# SmartLearn Agent - Product Design

## User Stories

1. As a ____________, I want to ____________, so that ____________.
2. As a ____________, I want to ____________, so that ____________.
3. As a ____________, I want to ____________, so that ____________.
```

#### Step 3: Fill in the user stories

Using the example above, complete the three blank user stories. Here are some ideas for inspiration (use these or write new ones):

| Role | Wants to | So that |
|------|---------|------|
| student | upload a PDF and ask questions about it | study more efficiently |
| student | get answers with page numbers | quickly find the original content in the PDF |
| student | ask follow-up questions in a conversation | deepen understanding of a topic |

Save the file when done (`Cmd + S` / `Ctrl + S`).

✅ **Checkpoint 1:** `docs/design.md` exists and contains 3 completed user stories.

---

## 4.12 Feature List + Timeline

### What Is a Feature List?

With user stories in place, the next step is to break them down into **concrete features**, rank them by priority, and decide what to build first.

### Priority Levels

Software teams commonly use P0/P1/P2 to mark priority:

| Priority | Meaning | Notes |
|--------|------|------|
| **P0** | Must have | The product is unusable without it. Must be done on Day 2. |
| **P1** | Should have | Makes things much better, though the basics work without it. Done on Day 3. |
| **P2** | Nice to have | Build if time allows; cut if it does not. Day 3 if there is time. |

### Hands-on: Add the Feature List to `docs/design.md`

Below the User Stories section in `docs/design.md`, add the following:

```markdown
## Feature List

| Priority | Feature | Day |
|----------|---------|-----|
| P0 | ______ | Day 2 |
| P0 | ______ | Day 2 |
| P1 | ______ | Day 3 |
| P1 | ______ | Day 3 |
| P2 | ______ | Day 3 |

## What We Will NOT Build

- ______
- ______
- ______
```

**Hints for filling it in:**

Some candidate features for reference:

| Priority | Feature | Day | Notes |
|----------|---------|-----|------|
| P0 | PDF text extraction | Day 2 | The foundation; nothing works without it |
| P0 | LLM Q&A with page citation | Day 2 | The core feature; users upload a PDF and ask questions |
| P1 | RAG pipeline | Day 3 | Handles long PDFs by retrieving only the most relevant parts |
| P1 | Web UI | Day 3 | Lets users interact through a browser |
| P2 | Chat history | Day 3 | Remembers earlier questions and supports follow-ups |

### Why "What We Will NOT Build" Matters

**Why list what to skip?** The biggest product risk is trying to do everything and finishing nothing. Explicitly listing what to skip is **scope control**.

Reference items:
- User authentication — workshop time is limited, so skip login
- Multi-file support — perfect the single-PDF experience first
- Mobile app — web version only

Save the file when done.

✅ **Checkpoint 2:** `docs/design.md` contains a completed feature list and a "What We Will NOT Build" section.

---

## 4.13 Data Flow Design

### What Is a Data Flow?

A **Data Flow Diagram** shows how data moves through the system — from input to output, and through which processing steps.

Think of it as the data's travel itinerary. A PDF file starts with the user upload, then passes through text extraction, prompt building, the AI call, and the returned answer — each step is a stop on the route.

### Why Draw It

Draw the data flow before coding, and you know which modules to write and how they pass data. Then implement the diagram step by step.

### Day 2 Data Flow (Simple Mode)

On Day 2 we build the simplest version: send the entire PDF text directly to the AI.

Add the following to `docs/design.md`, replacing each `[???]` with the processing step you think belongs there:

```markdown
## Data Flow

### Day 2: Simple Mode

PDF File
  -> [???]          # How do we get text out?
  -> pages[]
  -> [???]          # How do we combine with question?
  -> [LLM]
  -> Answer with [Page X]
```

**Hints:**

- First `[???]`: the step that pulls text out of the PDF. Which Python tool reads PDFs? (Hint: we used `PyPDF2` or a similar library in Section 4.4)
- Second `[???]`: combine the extracted text with the user's question into a prompt for the AI. What is this step called? (Hint: recall prompt engineering from Section 2)

**Reference answers (think first, then peek):**

> First: `[PDF parser / extract text]`
> Second: `[build prompt: pages + question]`

### Day 3 Data Flow (RAG Mode)

On Day 3 we handle longer PDFs. If a PDF has 200 pages, sending all of the text to the AI at once causes two problems:
1. AI input length is limited (the context window), and 200 pages of text may exceed it
2. Large amounts of irrelevant content degrade answer quality

**RAG (Retrieval-Augmented Generation)** works like this: split the PDF into small chunks, find the chunks most relevant to the question, and send only those to the AI.

Continue adding to `docs/design.md`:

```markdown
### Day 3: RAG Mode

PDF -> [???] -> pages
    -> [???] -> chunks with source_page
    -> [???] -> embeddings
    -> [???]  # storage

Question -> [encode] -> [???] -> relevant chunks -> [LLM] -> Answer
```

**Hints:**

The RAG flow has 5 key steps, matching the 5 `[???]` marks:

| Step | What It Does | Hint |
|------|--------|------|
| 1st `[???]` | Extract text from the PDF | Same as Day 2 |
| 2nd `[???]` | Split long text into small chunks | Called split / chunk |
| 3rd `[???]` | Turn text into numeric vectors | Called embed |
| 4th `[???]` | Store the vectors | Needs a vector store |
| 5th `[???]` | Search for the most relevant chunks | Called similarity search |

**Reference answers:**

> 1. `[extract text]`
> 2. `[split into chunks]`
> 3. `[embed]`
> 4. `[vector store (FAISS)]`
> 5. `[similarity search]`

### RAG in One Picture

```
Traditional: entire PDF text ──────────────────────> [LLM] -> answer
             (may be too long and exceed the AI input limit)

RAG:         entire PDF text -> chunk -> embed -> store
             user question -> embed -> search for the most relevant chunks -> [LLM] -> answer
             (sends only the most relevant parts; much more efficient)
```

On Day 3 you will implement this flow by hand — come back to this picture then and it will click.

Save the file when done.

✅ **Checkpoint 3:** `docs/design.md` contains the Day 2 and Day 3 data flow diagrams, with every `[???]` replaced by a concrete step name.

---

## 4.14 Commit `docs/design.md` to Git

The product design document is done — commit it to Git to save progress.

Run the following commands in the terminal, one at a time:

In [ ]:
# Stage docs/design.md
# git add docs/design.md

> **Expected output:** no output (silent success).

In [ ]:
# Commit to the local repository
# git commit -m "docs: product design"

> **Expected output:**
> ```
> [main xxxxxxx] docs: product design
>  1 file changed, XX insertions(+)
>  create mode 100644 docs/design.md
> ```

In [ ]:
# Push to GitHub
# git push

> **Expected output:**
> ```
> Enumerating objects: ...
> Counting objects: ...
> Writing objects: 100% ...
> To github.com:<username>/smartlearn-agent.git
>    xxxxxxx..xxxxxxx  main -> main
> ```

✅ **Checkpoint 4:** `docs/design.md` is committed and pushed to GitHub. Open the GitHub repository page and confirm that `docs/design.md` appears.

> **Troubleshooting:**
> - `nothing to commit` → The file may be unsaved. Go back to the editor, press `Cmd+S` (Mac) or `Ctrl+S` (Windows) to save, then run `git add` again
> - `git push` rejected → Run `git pull` first to fetch remote updates, then push again
> - `fatal: not a git repository` → You are outside the project folder; run `cd smartlearn-agent` to switch into it

---

## 4.15 Day 1 Final Checklist

Congratulations on completing all of Day 1! Before wrapping up, go through the checklist and confirm every item:

| Status | Item | Section |
|------|--------|-------------|
| [ ] | GitHub repo `smartlearn-agent` has 3+ commits | Section 1 |
| [ ] | `hello_llm.py` — first successful API call | Section 2 |
| [ ] | `experiments/prompt_lab.py` — three-prompt comparison experiment | Section 2 |
| [ ] | `CLAUDE.md` — project context and safety rules for Claude Code | Section 3 |
| [ ] | `cli_qa.py` — paste text, ask questions, get cited answers | Section 3 |
| [ ] | `pdf_summary.py` — vibe-coded PDF summary tool | Section 4 |
| [ ] | `docs/design.md` — user stories, features, data flow | Section 4 |
| [ ] | `.env` is **not** tracked by Git (listed in `.gitignore`) | Section 1 |

**How to verify:** run the following command in the terminal and confirm that all files exist:

In [ ]:
# ls hello_llm.py experiments/prompt_lab.py CLAUDE.md cli_qa.py pdf_summary.py docs/design.md

> **Expected output:** all file names are listed, with no `No such file or directory` errors.

Next, check that `.env` is untracked:

In [ ]:
# git status .env

> **Expected output:** `.env` should be absent from the list (it is ignored via `.gitignore`), or shown with a note that it is ignored.

**Confirm the commit count:**

In [ ]:
# git log --oneline

> **Expected output:** 3 or more commit records, similar to:
> ```
> xxxxxxx docs: product design
> xxxxxxx feat: pdf summary tool
> xxxxxxx feat: cli qa tool
> xxxxxxx docs: add Claude Code project context
> xxxxxxx feat: prompt engineering experiments
> xxxxxxx feat: first LLM API call
> xxxxxxx Initial commit
> ```

✅ **Checkpoint 5:** every item on the Day 1 checklist passes.

---

## 4.16 Homework

Before Day 2 starts, complete the following:

**Task 1: Browse the SmartLearn-AI reference repository**

Open the reference repository on GitHub and browse the complete codebase. It shows what Days 2–3 will build toward.

**Task 2: Write down 3 things you do not understand**

While browsing the reference repository, note 3 things that are unclear. For example:
- "What is `FAISS`?"
- "How do `embeddings` turn text into numbers?"
- "Why are the frontend and backend separated?"

These questions will be answered during Days 2–3. Learning with questions in mind beats passive listening.

---

## Appendix A: Workshop OpenRouter Model

| Model | Provider | Workshop use |
|---|---|---|
| `google/gemma-4-31b-it:free` | Gemma through OpenRouter | Default for the Python API exercises |

---


## Appendix B: Git Command Cheat Sheet

| Command | What It Does | When to Use |
|------|------|----------|
| `git status` | Show the current state | Check which files changed, at any time |
| `git add <file>` | Stage a file | Prepare to commit one file |
| `git add .` | Stage all changes | Prepare to commit everything |
| `git commit -m "message"` | Commit to the local repository | Save a progress snapshot |
| `git push` | Push to GitHub | Upload local commits to the remote |
| `git pull` | Pull from GitHub | Sync the latest remote code |
| `git log --oneline` | Show commit history | See which snapshots were saved |
| `git diff` | Show unstaged changes | See what has been modified |
| `git diff --staged` | Show staged changes | Confirm what will be committed after `git add` |
| `git checkout -- <file>` | Discard changes to a file | Restore the last committed state after a bad edit |

### Commit Message Format

```
type: description
```

| Type | Purpose | Example |
|------|------|------|
| `feat` | New feature | `feat: add PDF upload` |
| `fix` | Bug fix | `fix: handle empty PDF` |
| `docs` | Documentation change | `docs: update README` |
| `refactor` | Code refactoring | `refactor: extract PDF parser` |
| `style` | Formatting change | `style: fix indentation` |

---

## Appendix C: API Request & Response Format Reference

### Sending a Request

When calling the OpenRouter API, the request JSON looks like this:

```json
{
  "model": "google/gemma-4-31b-it:free",
  "messages": [
    {
      "role": "system",
      "content": "You are a helpful study assistant."
    },
    {
      "role": "user",
      "content": "What is machine learning?"
    }
  ]
}
```

| Field | Description |
|------|------|
| `model` | Which AI model to use |
| `messages` | The conversation history, as a list |
| `role: "system"` | Role instructions for the AI (optional) |
| `role: "user"` | What the user says |
| `role: "assistant"` | The AI's reply (used in multi-turn conversations) |

### Receiving a Response

The API returns JSON in this format:

```json
{
  "id": "chatcmpl-abc123",
  "choices": [
    {
      "message": {
        "role": "assistant",
        "content": "Machine learning is a subset of AI..."
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 25,
    "completion_tokens": 150,
    "total_tokens": 175
  }
}
```

| Field | Description |
|------|------|
| `choices[0].message.content` | The AI's answer text (the part you need most) |
| `finish_reason` | `"stop"` means normal completion; `"length"` means the output was truncated at the length limit |
| `usage.total_tokens` | Total tokens consumed by this request (relates to billing) |

### Extracting the AI's Answer in Python

```python
response = requests.post(url, headers=headers, json=payload)
data = response.json()
answer = data["choices"][0]["message"]["content"]
print(answer)
```

---

## Day 1 Complete! Tomorrow we start building the full SmartLearn Agent.

On Day 1 you set up a development environment from scratch, learned to call an AI API, write prompts, and use AI coding tools, and completed the product design.

Day 2 goes hands-on:
- Read PDF files with Python
- Build an AI assistant that answers questions about PDF content
- Set up a FastAPI backend + React frontend

Rest well — see you tomorrow!